# Final Project — Transfer Learning for Image Classification

This notebook implements the 10 requested tasks using TensorFlow/Keras and a VGG16 transfer-learning model. It contains code, not fabricated training outputs; run the cells in order in the JupyterLab environment that contains the project images.

**Dataset layout required** (folder names can be `validation`, `valid`, or `val`):

```text
waste_data/
├── train/
│   ├── class_1/ ... images ...
│   └── class_2/ ... images ...
├── validation/
│   ├── class_1/ ... images ...
│   └── class_2/ ... images ...
└── test/
    ├── class_1/ ... images ...
    └── class_2/ ... images ...
```

If your dataset is stored elsewhere, change `DATA_ROOT` in the configuration cell. Each split should have the same class subfolder names. The code supports two or more classes.


## Task 1 — Print the TensorFlow version

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)


## Dataset configuration and data generators

In [ ]:
# Update this path if your train/validation/test folders live elsewhere.
DATA_ROOT = Path("waste_data")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


def resolve_split_dirs(root: Path):
    """Return train, validation, and test directories under root."""
    val_names = ("validation", "valid", "val")
    train_dir = root / "train"
    test_dir = root / "test"
    val_dir = next((root / n for n in val_names if (root / n).is_dir()), None)
    if train_dir.is_dir() and test_dir.is_dir() and val_dir is not None:
        return train_dir, val_dir, test_dir
    return None


# Try the configured path and common folder names under the current working directory.
root_candidates = [DATA_ROOT, Path.cwd() / "data", Path.cwd() / "dataset", Path.cwd() / "waste_data", Path.cwd()]
resolved = None
checked = set()
for candidate in root_candidates:
    candidate = candidate.resolve()
    if candidate in checked:
        continue
    checked.add(candidate)
    resolved = resolve_split_dirs(candidate)
    if resolved:
        DATA_ROOT = candidate
        break

if resolved is None:
    raise FileNotFoundError(
        "Could not locate train, validation/valid/val, and test directories. "
        "Set DATA_ROOT to the folder containing those splits, then rerun this cell."
    )

train_dir, validation_dir, test_dir = resolved
print("Dataset root:", DATA_ROOT)
print("Training directory:", train_dir)
print("Validation directory:", validation_dir)
print("Test directory:", test_dir)

# VGG16's preprocessing function is applied consistently to all data splits.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
)
validation_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    str(train_dir), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True, seed=SEED
)
validation_generator = validation_datagen.flow_from_directory(
    str(validation_dir), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)


## Task 2 — Create `test_generator` using `test_datagen`

In [ ]:
test_generator = test_datagen.flow_from_directory(
    str(test_dir),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

print("Test images:", test_generator.samples)
print("Class indices:", test_generator.class_indices)


## Task 3 — Print the length of `train_generator`

In [ ]:
print("Length of train_generator (number of batches):", len(train_generator))


## Task 4 — Print the model summary

In [ ]:
num_classes = len(train_generator.class_indices)
if num_classes < 2:
    raise ValueError("Image classification requires at least two class subfolders in the training directory.")
if validation_generator.class_indices != train_generator.class_indices:
    raise ValueError("Training and validation class folders differ. Ensure that their class names match.")
if test_generator.class_indices != train_generator.class_indices:
    raise ValueError("Training and test class folders differ. Ensure that their class names match.")

vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
)
vgg_base.trainable = False

inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="image")
x = vgg_base(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.Dropout(0.30, name="dropout")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
extract_feat_model = Model(inputs, outputs, name="extract_feat_model")

extract_feat_model.summary()


## Task 5 — Compile the feature-extraction model

In [ ]:
extract_feat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print("Feature-extraction model compiled successfully.")


## Train the feature-extraction model

In [ ]:
EPOCHS_EXTRACT = 5
training_callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=1, min_lr=1e-6),
]

history_extract_feat = extract_feat_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_EXTRACT,
    callbacks=training_callbacks,
)


## Task 6 — Plot training and validation accuracy (`extract_feat_model`)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_extract_feat.history.get("accuracy", []), label="Training accuracy")
plt.plot(history_extract_feat.history.get("val_accuracy", []), label="Validation accuracy")
plt.title("Feature-Extraction Model: Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Fine-tune the VGG16 model

In [ ]:
# Reuse the feature-extraction model and unfreeze only VGG16's final convolutional block.
vgg_base.trainable = True
for layer in vgg_base.layers:
    layer.trainable = layer.name.startswith("block5_")

# Recompile after changing trainable layers, using a low learning rate for fine-tuning.
fine_tune_model = extract_feat_model
fine_tune_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS_FINE_TUNE = 3
fine_tune_history = fine_tune_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=training_callbacks,
)


## Task 7 — Plot training and validation loss (fine-tuned model)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(fine_tune_history.history.get("loss", []), label="Training loss")
plt.plot(fine_tune_history.history.get("val_loss", []), label="Validation loss")
plt.title("Fine-Tuned Model: Loss")
plt.xlabel("Epoch")
plt.ylabel("Categorical cross-entropy loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Task 8 — Plot training and validation accuracy (fine-tuned model)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(fine_tune_history.history.get("accuracy", []), label="Training accuracy")
plt.plot(fine_tune_history.history.get("val_accuracy", []), label="Validation accuracy")
plt.title("Fine-Tuned Model: Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Tasks 9 and 10 — Plot test image predictions from both models

In [ ]:
def plot_test_prediction(model, index_to_plot=1, title="Model prediction"):
    """Display one original test image plus its actual and predicted labels."""
    if not 0 <= index_to_plot < len(test_generator.filepaths):
        raise IndexError(
            f"index_to_plot={index_to_plot} is out of range for {len(test_generator.filepaths)} test images."
        )

    image_path = test_generator.filepaths[index_to_plot]
    display_image = load_img(image_path, target_size=IMG_SIZE)
    image_array = img_to_array(display_image)
    model_input = preprocess_input(np.expand_dims(image_array.copy(), axis=0))
    probabilities = model.predict(model_input, verbose=0)[0]
    predicted_idx = int(np.argmax(probabilities))
    actual_idx = int(test_generator.classes[index_to_plot])

    class_names = {idx: name for name, idx in test_generator.class_indices.items()}
    predicted_label = class_names[predicted_idx]
    actual_label = class_names[actual_idx]

    plt.figure(figsize=(6, 6))
    plt.imshow(display_image)
    plt.axis("off")
    plt.title(
        f"{title}\nActual: {actual_label} | Predicted: {predicted_label}\n"
        f"Confidence: {probabilities[predicted_idx]:.1%}"
    )
    plt.tight_layout()
    plt.show()
    print("Image:", image_path)
    print("Actual class:", actual_label)
    print("Predicted class:", predicted_label)
    print("Prediction probabilities:", {class_names[i]: float(p) for i, p in enumerate(probabilities)})


index_to_plot = 1


### Task 9 — Extract Features Model

In [ ]:
plot_test_prediction(
    extract_feat_model,
    index_to_plot=index_to_plot,
    title="Extract Features Model",
)


### Task 10 — Fine-Tuned Model

In [ ]:
plot_test_prediction(
    fine_tune_model,
    index_to_plot=index_to_plot,
    title="Fine-Tuned Model",
)


## Optional — Evaluate the fine-tuned model on the test set

In [ ]:
test_loss, test_accuracy = fine_tune_model.evaluate(test_generator, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
